# Эксперимент 22: Swin Transformer на мел-спектрограмме

**Статья:** Swin Transformer: Hierarchical Vision Transformer using Shifted Windows (Swin Transformer: иерархический vision transformer со сдвинутыми окнами) 2021

**Ссылка:** [https://arxiv.org/abs/2103.14030](https://arxiv.org/abs/2103.14030)

**Краткое описание модели:** Mel-спектрограмма -> ресайз/3-канальный ввод -> Swin Tiny + классификационная голова.

**Содержание статьи:** Иерархический window-attention в Swin позволяет эффективно моделировать дальний контекст при умеренной вычислительной цене. При переносе на аудио это работает как сильный трансформер-бейзлайн для спектральных карт. Эксперимент проверяет, насколько vision-transformer полезен для детекции дефектов речи.

In [1]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import time
from joblib import Parallel, delayed
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score, classification_report

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))
sys.path.insert(0, str(exp_dir))

from shared import config, data_utils, train_utils
from shared.results_utils import save_result_csv
from model import get_model

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
n_letters = letters_train.shape[1]
N_MELS, N_FRAMES = config.N_MELS, 320

def load_mel(p):
    m = data_utils.extract_mel_spectrogram(p, n_mels=N_MELS, max_frames=N_FRAMES)
    return m[np.newaxis, ...]
X_train = np.stack(Parallel(n_jobs=-1)(delayed(load_mel)(p) for p in paths_train))
X_val   = np.stack(Parallel(n_jobs=-1)(delayed(load_mel)(p) for p in paths_val))
X_test  = np.stack(Parallel(n_jobs=-1)(delayed(load_mel)(p) for p in paths_test))
# Ресайз до 224x224 и 3 канала для Swin
def to_swin_size(X):
    t = torch.from_numpy(X).float()
    t = F.interpolate(t, size=(224, 224), mode="bilinear", align_corners=False)
    t = t.repeat(1, 3, 1, 1)
    return t.numpy()
X_train = to_swin_size(X_train)
X_val   = to_swin_size(X_val)
X_test  = to_swin_size(X_test)
m, s = X_train.mean(axis=(0,2,3), keepdims=True), X_train.std(axis=(0,2,3), keepdims=True)
s = np.where(s < 1e-6, 1.0, s)
X_train = (X_train - m) / s
X_val   = (X_val - m) / s
X_test  = (X_test - m) / s
print("Swin input: (N, 3, 224, 224)")

Swin input: (N, 3, 224, 224)


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
train_ds = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(letters_train).float(), torch.from_numpy(y_train).long())
val_ds   = TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(letters_val).float(), torch.from_numpy(y_val).long())
test_ds  = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(letters_test).float(), torch.from_numpy(y_test).long())
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

model = get_model(num_classes=2, n_letters=n_letters, pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n_params}")

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Параметров: 27520908


In [4]:
weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = train_utils.get_lr_scheduler(optimizer, patience=config.LR_SCHEDULER_PATIENCE, factor=config.LR_SCHEDULER_FACTOR)
early_stopping = train_utils.EarlyStopping(patience=config.EARLY_STOPPING_PATIENCE)
best_ckpt_path = exp_dir / "best_ckpt.pt"
best_f1 = -1.0

def eval_loader(loader):
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for x, letters, y in loader:
            x, letters, y = x.to(device), letters.to(device), y.to(device)
            pred = model(x, letters).argmax(dim=1)
            all_p.append(pred.cpu().numpy())
            all_t.append(y.cpu().numpy())
    return np.concatenate(all_p), np.concatenate(all_t)

N_EPOCHS = 50
t0 = time.perf_counter()
for epoch in range(N_EPOCHS):
    model.train()
    losses = []
    for x, letters, y in train_loader:
        x, letters, y = x.to(device), letters.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, letters), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.DEFAULT_GRAD_CLIP)
        optimizer.step()
        losses.append(loss.item())
    vp, vt = eval_loader(val_loader)
    val_f1 = f1_score(vt, vp, average="macro")
    if val_f1 > best_f1:
        best_f1 = val_f1
        train_utils.save_best_checkpoint(model, best_ckpt_path)
    scheduler.step(val_f1)
    print(f"Epoch {epoch+1}/{N_EPOCHS}  loss={np.mean(losses):.4f}  val_f1_macro={val_f1:.4f}")
    if early_stopping.step(val_f1):
        print(f"Early stopping на эпохе {epoch+1}")
        break
train_time_sec = time.perf_counter() - t0
train_utils.load_best_checkpoint(model, best_ckpt_path, device)
print(f"Обучение: {train_time_sec:.1f} с. Лучший val_f1_macro={best_f1:.4f}")

Epoch 1/50  loss=0.6884  val_f1_macro=0.6122
Epoch 2/50  loss=0.6525  val_f1_macro=0.6360
Epoch 3/50  loss=0.6494  val_f1_macro=0.5992
Epoch 4/50  loss=0.6179  val_f1_macro=0.6690
Epoch 5/50  loss=0.5723  val_f1_macro=0.7689
Epoch 6/50  loss=0.5401  val_f1_macro=0.7121
Epoch 7/50  loss=0.4458  val_f1_macro=0.7406
Epoch 8/50  loss=0.3239  val_f1_macro=0.6560
Epoch 9/50  loss=0.2545  val_f1_macro=0.6943
Epoch 10/50  loss=0.2296  val_f1_macro=0.6944
Epoch 11/50  loss=0.1794  val_f1_macro=0.7023
Epoch 12/50  loss=0.0707  val_f1_macro=0.7125
Epoch 13/50  loss=0.0384  val_f1_macro=0.6933
Epoch 14/50  loss=0.0252  val_f1_macro=0.6856
Epoch 15/50  loss=0.0364  val_f1_macro=0.7327
Early stopping на эпохе 15
Обучение: 264.9 с. Лучший val_f1_macro=0.7689


In [ ]:
model.eval()
all_logits = []
with torch.no_grad():
    for x, letters, _ in test_loader:
        x, letters = x.to(device), letters.to(device)
        all_logits.append(model(x, letters).cpu().numpy())
logits = np.concatenate(all_logits)
y_pred = np.argmax(logits, axis=1)
y_proba = torch.softmax(torch.from_numpy(logits), dim=1).numpy()[:, config.CLASS_BAD]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, pos_label=config.CLASS_BAD, zero_division=0)
recall_bad = recall_score(y_test, y_pred, pos_label=config.CLASS_BAD, zero_division=0)
print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_22_swin_mel", 
    experiment_name="Swin on mel", 
    model="Swin Tiny", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes="mel resized 224x224, timm", 
    num_params=n_params, 
    train_time_sec=train_time_sec
)

              precision    recall  f1-score   support

        good       0.80      0.89      0.84       282
         bad       0.69      0.53      0.60       135

    accuracy                           0.77       417
   macro avg       0.74      0.71      0.72       417
weighted avg       0.76      0.77      0.76       417

Результат сохранён в result.csv текущего эксперимента
